# Telco Customer Churn — Classification Models
**BITS Pilani WILP · M.Tech (AIML/DSE) · Machine Learning · Assignment 2**
**Repository:** https://github.com/Karthekchakri/2025ac05112

This notebook is self-contained and runs end-to-end in **Google Colab**
(or any Jupyter environment, including the BITS Virtual Lab) — the dataset
is downloaded directly from an open-source online repository, no local
files required.

It trains and evaluates 5 classifiers on the **Telco Customer Churn** dataset:
1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbors Classifier
4. Gaussian Naive Bayes Classifier
5. Random Forest (Ensemble)

For each model it computes: **Accuracy, AUC, Precision, Recall, F1, MCC**,
and saves the fitted pipelines (`.joblib`) plus `test_data.csv` and
`metrics.json` — the same artifacts used by the Streamlit app (`app.py`)
in this repository.

In [ ]:
# If running in Google Colab, uncomment the line below to ensure packages are present
# !pip install -q scikit-learn pandas numpy matplotlib seaborn joblib kagglehub

import json
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Dataset — download from an open-source online repository

**Dataset:** Telco Customer Churn
**Also published on Kaggle as:** [`blastchar/telco-customer-churn`](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
**Originally distributed by:** IBM (public sample dataset, same file used across
many UCI-style churn-prediction tutorials and course assignments)

- 7,043 instances (assignment minimum: 500 ✅)
- 19 usable features after dropping the ID column (assignment minimum: 12 ✅)
- Binary target: `Churn` (Yes/No)

Three ways to get the exact same file — pick whichever works in your
environment. **Option A runs with zero setup** and is used by default below.

In [ ]:
# ---------------------------------------------------------------------------
# OPTION A (default, no credentials needed): direct URL download
# Works in Colab, BITS Virtual Lab, Streamlit Cloud, or any machine with
# internet access. This is the identical CSV distributed on Kaggle as
# "blastchar/telco-customer-churn".
# ---------------------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_URL)
print("Downloaded via direct URL. Shape:", df.shape)
df.head()

In [ ]:
# ---------------------------------------------------------------------------
# OPTION B (alternative): download straight from Kaggle via kagglehub
# Requires a free Kaggle account + API token (kaggle.json), which Colab
# will prompt you to upload the first time this runs. Uncomment to use.
# ---------------------------------------------------------------------------
# import kagglehub
# path = kagglehub.dataset_download("blastchar/telco-customer-churn")
# print("Kaggle dataset downloaded to:", path)
# df = pd.read_csv(f"{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv")
# print(df.shape)

In [ ]:
# ---------------------------------------------------------------------------
# OPTION C (alternative): download via the Kaggle CLI
# Requires `pip install kaggle` and a kaggle.json API token placed in
# ~/.kaggle/kaggle.json. Uncomment to use.
# ---------------------------------------------------------------------------
# !pip install -q kaggle
# !kaggle datasets download -d blastchar/telco-customer-churn -p ./data --unzip
# df = pd.read_csv("./data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
# print(df.shape)

In [ ]:
# TotalCharges has a handful of blank strings for customers with tenure == 0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Save the raw dataset locally too (satisfies "download the dataset file" requirement)
df.to_csv("telco.csv", index=False)
print("Saved raw dataset to telco.csv")

df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

target_col = "Churn"
feature_cols = [c for c in df.columns if c != target_col]

numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print(f"Total features: {len(feature_cols)} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
print(f"Total instances: {len(df)}")
print(f"Class balance:\n{df['Churn'].value_counts(normalize=True).round(3)}")

## 2. Train / test split

80% train / 20% held-out test, stratified on `Churn`.

In [ ]:
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Save the held-out TEST set only (Streamlit free tier has limited capacity,
# so only test data ships with the repo, per the assignment instructions)
test_export = X_test.copy()
test_export[target_col] = y_test.values
test_export.to_csv("test_data.csv", index=False)
print(f"Saved test_data.csv with {len(test_export)} rows")

## 3. Preprocessing pipeline

Shared `ColumnTransformer` (median-impute + scale numeric, most-frequent-impute + one-hot encode categorical) reused identically by every model.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ]
)

## 4. Train all 5 models and evaluate on the held-out test set

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=8),
    "kNN": KNeighborsClassifier(n_neighbors=15),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, max_depth=10
    ),
}

results = {}
fitted_pipelines = {}

os.makedirs("model", exist_ok=True)

for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe.named_steps["clf"], "predict_proba") else y_pred

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }
    results[name] = metrics
    fitted_pipelines[name] = pipe

    filename = "model/" + name.lower().replace(" ", "_").replace("(", "").replace(")", "") + ".joblib"
    joblib.dump(pipe, filename)

    print(f"\n{name}")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print(f"  -> saved to {filename}")

## 5. Comparison table (all 6 metrics, all 5 models)

In [ ]:
comparison_df = pd.DataFrame(results).T.round(3)
comparison_df

In [ ]:
with open("model/metrics.json", "w") as f:
    json.dump(results, f, indent=2)

metadata = {
    "target_col": target_col,
    "feature_cols": feature_cols,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
}
with open("model/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model/metrics.json and model/metadata.json")

## 6. Markdown comparison table for README.md

Copy the printed table below straight into the assignment README / submission PDF.

In [ ]:
print("| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |")
print("|---|---|---|---|---|---|---|")
for name, m in results.items():
    print(f"| {name} | {m['Accuracy']:.3f} | {m['AUC']:.3f} | {m['Precision']:.3f} | "
          f"{m['Recall']:.3f} | {m['F1']:.3f} | {m['MCC']:.3f} |")

## 7. Confusion matrices for all 5 models

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, pipe) in zip(axes, fitted_pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

## 8. Classification report — best model (Logistic Regression)

In [ ]:
best_pipe = fitted_pipelines["Logistic Regression"]
y_pred = best_pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

## 9. Observations

| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | Best overall balance of accuracy and AUC; a linear model that handles the mostly one-hot categorical feature space well without overfitting. Recall is its main weakness. |
| Decision Tree | Weakest model on every metric. A single tree overfits to specific split rules in this noisy dataset and generalises worse than the ensemble or linear alternatives. |
| kNN | Middle-of-the-pack performance. Works reasonably once numeric features are scaled, but the many one-hot categorical dimensions dilute the distance signal it relies on. |
| Naive Bayes | Lowest precision and accuracy, but by far the highest recall — it over-predicts churn due to the violated feature-independence assumption. Useful only where catching almost every churner matters more than avoiding false alarms. |
| Random Forest (Ensemble) | Highest precision and close to Logistic Regression on accuracy/AUC; more robust to noise than the single Decision Tree via bagging, though recall stays moderate. |
| **Overall Winner** | **Logistic Regression** — best AUC and F1 with a simple, fast, well-calibrated model, matched but not beaten by the heavier Random Forest ensemble on any metric. |

## 10. Next steps
- The `model/*.joblib` files saved here are the exact files the Streamlit app (`app.py`) loads.
- `telco.csv` (full raw dataset), `test_data.csv`, and `model/metrics.json` / `model/metadata.json`
  are all produced by this notebook.
- Repository: https://github.com/Karthekchakri/2025ac05112
- If running in Colab: download these files (or mount Google Drive) and copy them into
  the repo above alongside `app.py`, `requirements.txt`, and `README.md` before deploying
  to Streamlit Community Cloud.